# Phase E.1.h5 — Hybrid KAN at Akimbo v1.0 architecture

First real Phase E run. Hybrid topology: keep the (768×4hm → 1024) FT with factoriser + 4 input buckets + horizontal king mirror from Akimbo v1.0, but replace the entire v1.0 output stack (`screlu(16) → screlu(32) → Linear(1)`) with a single `ReluKAN(2048, 1) G=5 k=3` layer. Tests Phase C's hypothesis ("hybrid > full") and Q1 ("does ReLU-KAN's bell-shape advantage survive at the wider architecture") at the same time.

**Budget**: 800 superbatches × ~34 s/SB (from E.0 smoke) ≈ **7.6 h on A100, ~$9 on Colab Pro+**.

**Checkpointing**: `save_rate=50` → 16 checkpoints per run. Per the Phase E plan: pick the winning checkpoint by SPRT, never by training loss.

> **Crash fix (2026-06)**: long Colab runs were killing the browser tab with `Aw, Snap! SIGILL`. That is the Chrome renderer running out of memory because the run cell streamed every Bullet progress line (~48/superbatch) into the cell's DOM. The run cell below writes cargo output to a **log file only**, prints a tiny throttled heartbeat via `clear_output`, and **syncs each checkpoint to Drive as it lands** so a mid-run disconnect is not catastrophic. Do not revert it to streaming output.

## 1. Install Rust + clone repo

In [ ]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
set -e
rm -rf /content/bullet
cd /content
git clone https://github.com/y0sif/bullet.git
cd bullet
git log -1 --oneline
echo '---'
ls examples/phase_e1_h5.rs
grep -A1 'phase_e1_h5' crates/bullet_lib/Cargo.toml || echo 'WARN: phase_e1_h5 not in Cargo.toml'

## 2. Confirm A100 is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 3. Download January 2024 binpack (~7.7 GB compressed, ~25 GB decompressed)

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test80-2024-01-jan.binpack ]; then
    echo "Downloading test80-2024 January shard (~7.7 GB compressed)..."
    wget --progress=dot:giga -O test80-2024-01-jan.binpack.zst \
        "https://huggingface.co/datasets/linrock/test80-2024/resolve/main/test80-2024-01-jan-2tb7p.min-v2.v6.binpack.zst"
    echo "Decompressing..."
    zstd -d test80-2024-01-jan.binpack.zst -o test80-2024-01-jan.binpack --rm
fi

ls -lh test80-2024-01-jan.binpack
df -h /content

## 4. Build the E.1.h5 binary

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
cargo build --release --example phase_e1_h5 2>&1 | tail -20

## 5. Mount Drive (for incremental checkpoint sync)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/kanue/phase_e1_h5'
os.makedirs(DEST, exist_ok=True)
print('Checkpoints + log will sync to:', DEST)

## 6. Run training (~7.6 h) — output to log file, compact heartbeat, incremental Drive sync

**Do not** re-add per-line streaming to this cell — that is what crashed the browser tab. Output goes to `/content/phase_e1_h5_log.txt`; the cell shows only the latest superbatch line and refreshes it in place. Each checkpoint (`quantised.bin` + `raw.bin`) is copied to Drive as soon as it appears, so a disconnect loses at most the superbatches since the last 50-SB boundary.

In [ ]:
import subprocess, os, time, glob, shutil, re
from IPython.display import clear_output

os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

LOG_PATH = "/content/phase_e1_h5_log.txt"
CKPT_ROOT = "/content/bullet/checkpoints"
POLL_SECS = 30

if os.path.isdir(CKPT_ROOT):
    shutil.rmtree(CKPT_ROOT, ignore_errors=True)

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

def latest_status(path):
    """Last non-empty chunk of the log, splitting on \r (progress bar) and \n."""
    try:
        with open(path, 'rb') as f:
            f.seek(0, 2)
            size = f.tell()
            f.seek(max(0, size - 8192))
            tail = f.read().decode('utf-8', 'ignore')
    except FileNotFoundError:
        return "(log not created yet)"
    chunks = [c.strip() for c in re.split(r'[\r\n]', strip_ansi(tail)) if c.strip()]
    return chunks[-1] if chunks else "(no output yet)"

def sync_new_checkpoints():
    """Copy quantised.bin + raw.bin for any checkpoint not yet on Drive."""
    copied = []
    for ckpt in sorted(glob.glob(os.path.join(CKPT_ROOT, 'phase_e1_h5-*'))):
        name = os.path.basename(ckpt)
        out_dir = os.path.join(DEST, name)
        if os.path.isdir(out_dir):
            continue  # already synced
        os.makedirs(out_dir, exist_ok=True)
        for fname in ['quantised.bin', 'raw.bin']:
            src = os.path.join(ckpt, fname)
            if os.path.isfile(src):
                shutil.copy(src, out_dir)
        copied.append(name)
    return copied

logf = open(LOG_PATH, 'w')
start = time.time()
proc = subprocess.Popen(
    ["cargo", "run", "--release", "--example", "phase_e1_h5"],
    stdout=logf, stderr=subprocess.STDOUT, cwd="/content/bullet", text=True,
)

synced = []
loop = 0
while proc.poll() is None:
    time.sleep(POLL_SECS)
    synced += sync_new_checkpoints()
    # copy the log to Drive every ~5 min as a cheap safety net
    loop += 1
    if loop % 10 == 0:
        try:
            shutil.copy(LOG_PATH, DEST)
        except Exception:
            pass
    elapsed = time.time() - start
    clear_output(wait=True)
    print(f"[E.1.h5] running  |  elapsed {elapsed/3600:.2f} h")
    print(f"latest : {latest_status(LOG_PATH)}")
    print(f"synced : {len(synced)} checkpoints -> {synced}")

logf.close()
synced += sync_new_checkpoints()
try:
    shutil.copy(LOG_PATH, DEST)
except Exception:
    pass

elapsed = time.time() - start
clear_output(wait=True)
print(f"[E.1.h5] DONE  exit={proc.returncode}  elapsed {elapsed/3600:.2f} h")
print(f"latest : {latest_status(LOG_PATH)}")
on_drive = sorted(os.path.basename(p) for p in glob.glob(os.path.join(DEST, 'phase_e1_h5-*')))
print(f"checkpoints on Drive ({len(on_drive)}): {on_drive}")
if proc.returncode != 0:
    print("\nWARNING: training exited non-zero. Inspect the log on Drive; checkpoints already synced are still usable.")

## 7. Parse loss + throughput, plot loss curve

Reads the log file (also on Drive). The plot DOM cost is tiny — one image, not 40k lines — so this cell is safe to run.

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

LOG_PATH = "/content/phase_e1_h5_log.txt"
loss_records = []
time_records = []

with open(LOG_PATH) as f:
    raw = strip_ansi(f.read())
# the progress bar uses \r within a superbatch; the summary line ends each one.
for line in re.split(r'[\r\n]', raw):
    m_loss = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', line)
    if m_loss:
        loss_records.append((int(m_loss.group(1)), float(m_loss.group(2))))
    m_time = re.search(r'superbatch\s+(\d+)\s+\|.*?time\s+(\d+\.\d+)s', line)
    if m_time:
        time_records.append((int(m_time.group(1)), float(m_time.group(2))))

if not loss_records:
    print("WARNING: no loss lines parsed.")
else:
    first_sb, first_loss = loss_records[0]
    last_sb, last_loss = loss_records[-1]
    print(f"Loss: SB {first_sb}  {first_loss:.6f}  ->  SB {last_sb}  {last_loss:.6f}")
    print(f"Relative drop: {(first_loss - last_loss) / first_loss * 100:.1f}%")
    print(f"Completed {last_sb}/800 superbatches")

if time_records:
    secs = [t for _, t in time_records]
    mean_sb = sum(secs) / len(secs)
    print(f"\nMean per-superbatch wall-clock: {mean_sb:.1f} s")
    print(f"Total training time: {mean_sb * len(secs) / 3600:.2f} h")

if loss_records:
    fig, ax = plt.subplots(figsize=(12, 5))
    xs = [sb for sb, _ in loss_records]
    ys = [l for _, l in loss_records]
    ax.plot(xs, ys, linewidth=1.5)
    ax.set_xlabel("Superbatch")
    ax.set_ylabel("Running loss")
    ax.set_title("Phase E.1.h5 — Hybrid ReLU-KAN at v1.0 architecture")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out_png = "/content/phase_e1_h5_loss.png"
    plt.savefig(out_png, dpi=150)
    plt.show()
    import shutil, os
    if os.path.isdir('/content/drive/MyDrive/kanue/phase_e1_h5'):
        shutil.copy(out_png, '/content/drive/MyDrive/kanue/phase_e1_h5')